# Evaluate Performance Using SVM Regression (RBF Kernel) 

In [20]:
# Import statements (add as needed)

import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, cross_validate, train_test_split
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler, FunctionTransformer, PowerTransformer
from sklearn.impute import SimpleImputer

from sklearn.compose import (
    TransformedTargetRegressor,
    make_column_transformer,
)

from sklearn.pipeline import make_pipeline
from skopt import BayesSearchCV
from missforest import MissForest

from sklearn.multioutput import MultiOutputRegressor
from sklearn.multioutput import RegressorChain
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from shap_select import shap_select
import xgboost as xgb
from xgboost import XGBRegressor

In [3]:
# load both feature engineered dataset.

fe_df = pd.read_csv('../data/feature_engineered_training_set.csv')

In [4]:
fe_df = fe_df.drop(columns=['Unnamed: 0'])
fe_df.head()

,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,pet,elevation,EVI,NDVI,Land Surface Temperature,nir,green,...,total_precipitation_sum,volumetric_soil_water,precipitation,cec_pH_interaction,phosphorous_pH_interaction,NDVI_LST_interaction,flow_acc_clay_interaction,flow_acc_phosphorous_interaction,evaporation_precipitation_ratio,cec_clay_ratio
0,128.912,555.0,10.0,174.2,167.155040,241.0,636.0,15845.0,11190.0,11426.0,...,0.000001,0.008799,0.488024,1800.0,1725.0,10077420.0,8.676030e+07,9.502319e+07,-37.216627,1.142857
1,74.720,162.9,163.0,124.1,1521.251493,4643.0,7656.0,15190.0,17658.5,9550.0,...,0.007830,0.458929,97.342756,1755.0,1365.0,116294640.0,3.369889e+05,2.440265e+05,-0.523901,0.931034
2,89.254,573.0,80.0,127.5,1471.379902,3984.0,6276.0,14879.0,15210.0,10720.0,...,0.006964,0.393498,96.911494,1472.0,1344.0,93380604.0,2.800000e+01,2.100000e+01,-0.575803,0.821429
3,82.000,203.6,101.0,129.7,1342.659998,2217.0,3957.0,15228.5,14887.0,10943.0,...,0.007228,0.409018,102.307634,1495.0,1430.0,60259174.5,4.884100e+05,4.132700e+05,-0.538889,0.884615
4,56.100,145.1,151.0,129.2,1355.983661,4136.0,7396.0,15130.0,16828.5,9502.5,...,0.004980,0.437815,96.513882,1600.0,1280.0,111901480.0,1.337840e+05,9.556000e+04,-0.784851,0.892857


**SVM is quite sensitive to scaling, hence scale with standard scaler. Lets see if feature engineering helped during feature selection**

In [5]:
train_df, test_df = train_test_split(fe_df, test_size=0.3)      # create train set and test set to eval performance

In [6]:
train_df.head()

,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,pet,elevation,EVI,NDVI,Land Surface Temperature,nir,green,...,total_precipitation_sum,volumetric_soil_water,precipitation,cec_pH_interaction,phosphorous_pH_interaction,NDVI_LST_interaction,flow_acc_clay_interaction,flow_acc_phosphorous_interaction,evaporation_precipitation_ratio,cec_clay_ratio
2662,177.103,452.0,10.0,154.20000,1496.121764,1131.0,2365.0,15121.0,NaN,NaN,...,1.722232e-06,0.257460,0.000000,1716.0,1188.0,35761165.00,2.357606e+06,1.463341e+06,-40.286367,0.896552
2649,81.079,243.8,10.0,134.90001,232.428267,3100.0,5526.0,14860.0,18055.0,9539.5,...,4.234621e-04,0.286886,0.000000,1500.0,1620.0,82116360.00,3.978163e+04,3.703807e+04,-5.702253,0.862069
1835,93.680,186.9,10.0,170.80000,1295.731321,2824.0,5599.0,15227.0,NaN,NaN,...,8.240361e-06,0.229584,14.154877,1820.0,1540.0,85255973.00,3.653896e+06,2.870918e+06,-443.982536,0.928571
8420,163.440,591.0,20.0,182.40001,1521.251493,1493.0,2598.5,14735.5,14232.0,10420.5,...,8.583069e-07,0.204995,0.000000,1755.0,1365.0,38290196.75,3.369889e+05,2.440265e+05,-84.782466,0.931034
2670,146.398,672.0,115.0,173.80000,1320.711972,1525.0,3340.5,15017.0,9872.0,9824.0,...,1.938298e-06,0.200379,7.927866,1342.0,1342.0,50164288.50,6.188652e+06,5.672931e+06,-29.499135,0.916667


In [7]:
train_df.shape

(6523, 33)

### Preprocess Data (Scale + Impute)

In [8]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6523 entries, 2662 to 488
Data columns (total 33 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Total Alkalinity                  6523 non-null   float64
 1   Electrical Conductance            6523 non-null   float64
 2   Dissolved Reactive Phosphorus     6523 non-null   float64
 3   pet                               6523 non-null   float64
 4   elevation                         6523 non-null   float64
 5   EVI                               6523 non-null   float64
 6   NDVI                              6523 non-null   float64
 7   Land Surface Temperature          6523 non-null   float64
 8   nir                               5790 non-null   float64
 9   green                             5790 non-null   float64
 10  swir16                            5790 non-null   float64
 11  swir22                            5790 non-null   float64
 12  NDMI     

In [9]:
train_df.columns[train_df.isnull().any()].tolist()

['nir',
 'green',
 'swir16',
 'swir22',
 'NDMI',
 'MNDWI',
 'skin_temperature',
 'soil_temperature',
 'temperature_2m',
 'total_evaporation_sum',
 'total_precipitation_sum',
 'volumetric_soil_water',
 'precipitation',
 'evaporation_precipitation_ratio']

Preprocessing must handle missing values, then scale all data. Can be done in a pipeline. Scaling for all features, imputation only for those with missing values. 

In [10]:
## Data to be imputed with iterative imputer using random forest estimator to approximate miss forest.

imputing_feats = ['nir',
 'green',
 'swir16',
 'swir22',
 'NDMI',
 'MNDWI',
 'skin_temperature',
 'soil_temperature',
 'temperature_2m',
 'total_evaporation_sum',
 'total_precipitation_sum',
 'volumetric_soil_water',
 'precipitation',
 'evaporation_precipitation_ratio']


In [11]:
# Split train_df and test_df into X_train, X_test, y_train and y_test

X_train = train_df.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])
y_train = train_df[['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']]

X_test = test_df.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])
y_test = test_df[['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']]

In [12]:
X_train.head()

,pet,elevation,EVI,NDVI,Land Surface Temperature,nir,green,swir16,swir22,NDMI,...,total_precipitation_sum,volumetric_soil_water,precipitation,cec_pH_interaction,phosphorous_pH_interaction,NDVI_LST_interaction,flow_acc_clay_interaction,flow_acc_phosphorous_interaction,evaporation_precipitation_ratio,cec_clay_ratio
2662,154.20000,1496.121764,1131.0,2365.0,15121.0,NaN,NaN,NaN,NaN,NaN,...,1.722232e-06,0.257460,0.000000,1716.0,1188.0,35761165.00,2.357606e+06,1.463341e+06,-40.286367,0.896552
2649,134.90001,232.428267,3100.0,5526.0,14860.0,18055.0,9539.5,14137.0,10647.0,0.121707,...,4.234621e-04,0.286886,0.000000,1500.0,1620.0,82116360.00,3.978163e+04,3.703807e+04,-5.702253,0.862069
1835,170.80000,1295.731321,2824.0,5599.0,15227.0,NaN,NaN,NaN,NaN,NaN,...,8.240361e-06,0.229584,14.154877,1820.0,1540.0,85255973.00,3.653896e+06,2.870918e+06,-443.982536,0.928571
8420,182.40001,1521.251493,1493.0,2598.5,14735.5,14232.0,10420.5,17884.5,14877.0,-0.113727,...,8.583069e-07,0.204995,0.000000,1755.0,1365.0,38290196.75,3.369889e+05,2.440265e+05,-84.782466,0.931034
2670,173.80000,1320.711972,1525.0,3340.5,15017.0,9872.0,9824.0,9109.0,8584.0,0.040198,...,1.938298e-06,0.200379,7.927866,1342.0,1342.0,50164288.50,6.188652e+06,5.672931e+06,-29.499135,0.916667


In [14]:
# Create a pipeline to first impute and then scale data.

preprocessor = make_pipeline(
    IterativeImputer(RandomForestRegressor(n_estimators=100, n_jobs=-1)),
    StandardScaler())

## **Feature Selection** 

In [25]:
# Conduct Feature selection using shap-select.
# Convert X_train, y_train, X_test, y_test to dMatrix for quicker computation with XGBoost.

X_tr, X_te, y_tr, y_te = train_test_split(X_train, y_train, test_size=0.2)# split X and y train to conduct feature selection with less risk of overfitting.

imputer = IterativeImputer(RandomForestRegressor(n_estimators=100, n_jobs=-1))

X_tr_fe = imputer.fit_transform(X_tr)
X_te_fe = imputer.transform(X_te)

model = XGBRegressor(n_estimators=1000, verbosity = 0, eval_metric='rmse', objective='reg:squarederror')
model.fit(X_tr_fe, y_tr, eval_set=[(X_te_fe, y_te)])

[0]	validation_0-rmse:168.04381
[1]	validation_0-rmse:146.59254
[2]	validation_0-rmse:130.73105
[3]	validation_0-rmse:120.45616
[4]	validation_0-rmse:111.20117
[5]	validation_0-rmse:106.47870
[6]	validation_0-rmse:101.55609
[7]	validation_0-rmse:98.60203
[8]	validation_0-rmse:96.67591
[9]	validation_0-rmse:94.77552
[10]	validation_0-rmse:93.43011
[11]	validation_0-rmse:92.33267
[12]	validation_0-rmse:91.23877
[13]	validation_0-rmse:90.66998
[14]	validation_0-rmse:89.78317
[15]	validation_0-rmse:89.52940
[16]	validation_0-rmse:88.83717
[17]	validation_0-rmse:88.64103
[18]	validation_0-rmse:88.31201
[19]	validation_0-rmse:87.69775
[20]	validation_0-rmse:87.75310
[21]	validation_0-rmse:87.49676
[22]	validation_0-rmse:87.00309
[23]	validation_0-rmse:86.94456
[24]	validation_0-rmse:87.10972
[25]	validation_0-rmse:86.95969
[26]	validation_0-rmse:86.79934
[27]	validation_0-rmse:85.77407
[28]	validation_0-rmse:85.90435
[29]	validation_0-rmse:85.83699
[30]	validation_0-rmse:85.94894
[31]	valida

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric='rmse', feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=1000,
             n_jobs=None, num_parallel_tree=None, ...)

In [45]:
# Next steps: Scale data and perform hyperparameter optimization

pipe = make_pipeline(StandardScaler(), MultiOutputRegressor(SVR()))
search_space = {
    "multioutputregressor__estimator__C": np.logspace(0, 2, 10),
    "multioutputregressor__estimator__gamma": 10.0 ** np.logspace(-3, 0, 10),
    "multioutputregressor__estimator__epsilon": np.logspace(-3, 0, 10)
}

search = BayesSearchCV(pipe, search_spaces=search_space, cv=10, return_train_score=True)
search.fit(X_train_rfe, y_train)

BayesSearchCV(cv=10,
              estimator=Pipeline(steps=[('standardscaler', StandardScaler()),
                                        ('multioutputregressor',
                                         MultiOutputRegressor(estimator=SVR()))]),
              return_train_score=True,
              search_spaces={'multioutputregressor__estimator__C': array([  1.        ,   1.66810054,   2.7825594 ,   4.64158883,
         7.74263683,  12.91549665,  21.5443469 ,  35.93813664,
        59.94842503, 100.        ]),
                             'multioutputregressor__estimator__epsilon': array([0.001     , 0.00215443, 0.00464159, 0.01      , 0.02154435,
       0.04641589, 0.1       , 0.21544347, 0.46415888, 1.        ]),
                             'multioutputregressor__estimator__gamma': array([ 1.00230524,  1.00497309,  1.01074497,  1.02329299,  1.05085876,
        1.11279685,  1.25892541,  1.64226588,  2.91178218, 10.        ])})

In [50]:
pd.DataFrame(search.cv_results_)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_multioutputregressor__estimator__C,param_multioutputregressor__estimator__epsilon,param_multioutputregressor__estimator__gamma,params,split0_test_score,split1_test_score,...,split3_train_score,split4_train_score,split5_train_score,split6_train_score,split7_train_score,split8_train_score,split9_train_score,mean_train_score,std_train_score,rank_train_score
0,1.582671,0.042356,0.304575,0.008335,7.742637,0.002154,1.010745,{'multioutputregressor__estimator__C': 7.74263...,0.086708,0.043829,...,0.169461,0.169616,0.170576,0.168989,0.171299,0.169995,0.170697,0.169459,0.001849,38
1,1.561258,0.017597,0.311083,0.005713,4.641589,0.004642,1.023293,{'multioutputregressor__estimator__C': 4.64158...,0.025693,-0.012033,...,0.074099,0.075233,0.075764,0.074198,0.076496,0.074741,0.075953,0.074958,0.001388,42
2,1.488706,0.031019,0.308189,0.006614,1.000000,0.002154,2.911782,"{'multioutputregressor__estimator__C': 1.0, 'm...",-0.073953,-0.104523,...,-0.072780,-0.071753,-0.071788,-0.071822,-0.071609,-0.073005,-0.071716,-0.071921,0.000605,50
3,1.577950,0.032292,0.305206,0.006029,7.742637,0.021544,1.642266,{'multioutputregressor__estimator__C': 7.74263...,0.013309,-0.028764,...,0.102287,0.103257,0.103419,0.102695,0.104507,0.102391,0.103726,0.102773,0.001454,41
4,1.393734,0.008978,0.288003,0.001420,1.000000,0.100000,1.642266,"{'multioutputregressor__estimator__C': 1.0, 'm...",-0.066815,-0.098045,...,-0.066105,-0.065083,-0.065059,-0.065288,-0.064770,-0.066132,-0.064857,-0.065241,0.000608,49
5,1.806105,0.087685,0.311249,0.010119,21.544347,0.464159,1.010745,{'multioutputregressor__estimator__C': 21.5443...,0.260762,0.214463,...,0.449641,0.447490,0.449922,0.450667,0.450451,0.449412,0.451393,0.448558,0.002632,32
6,1.710354,0.016773,0.302228,0.004521,12.915497,0.001000,1.112797,{'multioutputregressor__estimator__C': 12.9154...,0.144971,0.096487,...,0.276872,0.276172,0.277204,0.276098,0.278631,0.277058,0.278218,0.276272,0.002338,36
7,2.162698,0.029628,0.311185,0.004723,59.948425,0.100000,10.000000,{'multioutputregressor__estimator__C': 59.9484...,0.039879,0.006691,...,0.598565,0.596552,0.596963,0.599801,0.599249,0.597284,0.598927,0.597844,0.001281,27
8,1.570883,0.025779,0.302421,0.004542,7.742637,0.001000,1.258925,{'multioutputregressor__estimator__C': 7.74263...,0.048369,0.005425,...,0.133912,0.134856,0.135374,0.134141,0.136343,0.134235,0.135409,0.134400,0.001656,40
9,2.045654,0.022148,0.303204,0.005257,35.938137,0.002154,1.010745,{'multioutputregressor__estimator__C': 35.9381...,0.355781,0.311600,...,0.605819,0.602874,0.606087,0.606946,0.605782,0.605228,0.607010,0.604591,0.002102,23


In [51]:
search.best_score_

np.float64(0.4754122308294789)

In [52]:
search.best_params_

OrderedDict([('multioutputregressor__estimator__C', 100.0),
             ('multioutputregressor__estimator__epsilon',
              0.004641588833612777),
             ('multioutputregressor__estimator__gamma', 1.0023052380778996)])

Looks like the best parameters for SVR (which by the looks of it is performing quite poorly) are:

    - C: 100
    - epsilon: 0.005
    - gamma: 1

In [57]:
# Test performance with Regressor Chaining, given the optimized hyperparamters.
from itertools import permutations

targets = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']
combinations = list(permutations(range(len(targets))))

# alkalinity = 0, electrical conductance = 1, dissolved reactive phosphorous = 2
combinations

[(0, 1, 2), (0, 2, 1), (1, 0, 2), (1, 2, 0), (2, 0, 1), (2, 1, 0)]

In [67]:
# Test with all possible chain combinations!

svr = SVR(C=100, epsilon=0.005, gamma=1)
scores = []

for combo in combinations:
    chain = RegressorChain(base_estimator=svr, order=combo)
    pipe = make_pipeline(StandardScaler(), chain)

    cv_scores = cross_validate(pipe, X_train_rfe, y_train, cv=5, return_train_score=True, n_jobs=-1)

    scores.append({
        'order': combo,
        'mean_train_score': np.mean(cv_scores['train_score']),
        'mean_test_score': np.mean(cv_scores['test_score']),
        "mean_fit_time": np.mean(cv_scores["fit_time"]),
        "mean_score_time": np.mean(cv_scores["score_time"])
    })

In [68]:
scores_df = pd.DataFrame(scores)
scores_df = scores_df.sort_values(by='mean_test_score', ascending=False)
scores_df

,order,mean_train_score,mean_test_score,mean_fit_time,mean_score_time
1,"(0, 2, 1)",0.686311,0.190337,3.436365,1.794748
0,"(0, 1, 2)",0.461263,0.177547,2.865268,1.673056
4,"(2, 0, 1)",0.694198,0.153516,4.049290,2.103889
5,"(2, 1, 0)",0.490149,0.141511,5.563361,2.744834
2,"(1, 0, 2)",0.332381,0.115743,3.075551,1.814276
3,"(1, 2, 0)",0.334365,0.115688,3.324587,1.825534


Looks like MultiOutputRegression worked much better than Regression Chaining, lets use that instead.

In [ ]:
## FINAL MODEL EVALUATION ON TEST SET TO SEE PERFORMANCE

svr_final = make_pipeline(StandardScaler(), MultiOutputRegressor(SVR(C=100, epsilon=0.005, gamma=1)))
svr_final.fit(X_train_rfe, y_train)
test_score = svr_final.score(X_test_rfe, y_test)
print(test_score)

0.49111691799167495


In [85]:
## Predict Output for Submission Set

submission_df = pd.read_csv('../data/validation_set.csv')
submission_df.head()

,Unnamed: 0,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,pet,elevation,EVI,...,total_precipitation_sum,volumetric_soil_water,precipitation,cec_pH_interaction,phosphorous_pH_interaction,NDVI_LST_interaction,flow_acc_clay_interaction,flow_acc_phosphorous_interaction,evaporation_precipitation_ratio,cec_clay_ratio
0,0,-32.043333,27.822778,01-09-2014,NaN,NaN,NaN,161.90001,780.299056,1628.0,...,0.000007,0.136892,0.000000,1742.0,1742.0,4.542096e+07,351115.714286,338111.428571,-95.760822,0.962963
1,1,-33.329167,26.077500,16-09-2015,NaN,NaN,NaN,177.60000,279.351413,2933.0,...,0.000550,0.259777,27.521283,1495.0,1690.0,1.069767e+08,202225.000000,210314.000000,-3.653135,0.920000
2,2,-32.991639,27.640028,07-05-2015,NaN,NaN,NaN,158.40001,163.622679,4032.0,...,0.000174,0.232120,14.464016,1403.0,1525.0,1.034820e+08,121744.000000,126816.666667,-10.921542,0.958333
3,3,-34.096389,24.439167,07-02-2012,NaN,NaN,NaN,130.00000,44.046365,4701.0,...,NaN,NaN,42.085633,1121.0,1416.0,1.123978e+08,18889.531915,18889.531915,NaN,0.791667
4,4,-32.000556,28.581667,01-10-2014,NaN,NaN,NaN,152.50000,333.249422,1498.0,...,0.000618,0.336999,30.467072,1674.0,1674.0,6.468512e+07,410099.609756,381816.878049,-4.473473,0.931034


In [ ]:
imp_df = submission_df.drop(columns=['Longitude', 'Latitude', 
                                     'Unnamed: 0', 'Sample Date',
                                      'Total Alkalinity', 'Electrical Conductance',
                                      'Dissolved Reactive Phosphorus'])
sumbission_imp_df = mf.transform(imp_df)
imp_df.shape

In [79]:
submission_feats = ['flow_acc_phosphorous_interaction', 'flow_acc_clay_interaction', 'phosphorous_pH_interaction',
               'cec_pH_interaction', 'flow_accumulation', 'phosphorous', 'cec_clay_ratio', 'pet', 'elevation',
               'Land Surface Temperature', 'NDVI', 'volumetric_soil_water', 'total_evaporation_sum', 'MNDWI',
                'swir22', 'clay', 'cec', 'pH', 'EVI', 'soil_temperature', 'evaporation_precipitation_ratio', 
                'nir', 'green', 'NDMI', 'Latitude', 'Longitude', 'Sample Date']

submission_df = submission_df[submission_feats]
submission_df.columns

Index(['flow_acc_phosphorous_interaction', 'flow_acc_clay_interaction',
       'phosphorous_pH_interaction', 'cec_pH_interaction', 'flow_accumulation',
       'phosphorous', 'cec_clay_ratio', 'pet', 'elevation',
       'Land Surface Temperature', 'NDVI', 'volumetric_soil_water',
       'total_evaporation_sum', 'MNDWI', 'swir22', 'clay', 'cec', 'pH', 'EVI',
       'soil_temperature', 'evaporation_precipitation_ratio', 'nir', 'green',
       'NDMI', 'Latitude', 'Longitude', 'Sample Date'],
      dtype='object')

In [81]:
submission_data = submission_df.drop(columns=['Longitude', 'Latitude', 'Sample Date'])

In [82]:
submission_df.shape

(200, 27)

Edit: so this wont actually work because i trained MissForest on the missing target values now and so if i dont feed it target values it cant transform any data and since SVM cant natively handle Nan values there isnt really any way around this without restarting from the very beginning. Since the model performance is quite poor, I have decided to scrap it entirely.